# Iterative Proportional Representation – Demonstration

This notebook demonstrates the iterative electoral procedure using small examples that can be verified with a pocket calculator.
It covers only the voting process – no seat allocation or constituency assignment.

Concept explanation: [iterative_proportional_election.md](../../docs/source/en/iterative_proportional_election.md)

## Setup

**Key principle of the examples:** 1 constituency, 100 votes total → vote count = percentage.

By default the simulation generates votes randomly. In this notebook, fixed votes are injected instead to make the examples verifiable by hand:

- First round: `election.start(votes={'A': 28, ...})`
- Subsequent rounds: `round.getNextRoundInput().with_votes({'A': 35, ...})`

In [ ]:
import pandas as pd

from ipres import (
    Election, ElectionConfig, ElectionRound,
    Ballot, DrawOfLots, DrawLotsStrategy,
    SuperMajorityMargin, MarginUnit,
)
from ipres.constituencies_config import ConstituenciesConfig

# 1 constituency, 100 votes
cc = ConstituenciesConfig.from_dataframe(pd.DataFrame({
    'constituency_name': ['C1'],
    'constituency_size': [100],
    'turnout_percent':   [100.0],
}))

---
## Example 1: Reduction procedure (3 rounds)

Five parties compete. Winning threshold: **52 %** (majority margin 2 %).

| Party | Round 1 | Cumulative | Round 2 | Cumulative | Round 3 |
|-------|:-------:|:----------:|:-------:|:----------:|:-------:|
| A     | 28      | 28 %       | 35      | 35 %       | **52 ✓** |
| B     | 25      | 53 %       | 33      | 68 % ≥ ⅔   | 48      |
| C     | 20      | 73 % ≥ ⅔   | 32      | —          | —       |
| D     | 15      | —          | —       | —          | —       |
| E     | 12      | —          | —       | —          | —       |

*Reduction rule:* Parties are sorted by vote share in descending order and cumulated until ⅔ of all votes is reached.
All parties not included in this running total are eliminated.

In [ ]:
config = ElectionConfig(
    constituencies_config=cc,
    participating_parties=['A', 'B', 'C', 'D', 'E'],
    parliament_majority_margin=SuperMajorityMargin(2.0, MarginUnit.PERCENT),
    draw_lots_strategy=DrawLotsStrategy.MARGINAL_LEAD,
)

election = Election(electionConfig=config)

# --- Round 1 ---
round1 = election.start(votes={'A': 28, 'B': 25, 'C': 20, 'D': 15, 'E': 12})

print("=== Round 1 ===")
print(f"Threshold: {config.getParliamentMajorityPercent():.0f} %")
print(round1.getContestantsVotesAfterPossibleCoalitions().to_string())
print(f"\nWinner: {round1.getWinner()}")
print(f"Next round: {round1.hasNext()}")
print(f"Remaining: {list(round1.getNextRoundInput().contestants.keys())}")
print("  → D (15%) and E (12%) eliminated: A+B+C = 73% ≥ 66.7% (⅔)")

In [ ]:
# --- Round 2 (A, B, C) ---
round2 = election.runNextIteration(
    iterationInput=round1.getNextRoundInput().with_votes({'A': 35, 'B': 33, 'C': 32})
)

print("=== Round 2 ===")
print(round2.getContestantsVotesAfterPossibleCoalitions().to_string())
print(f"\nWinner: {round2.getWinner()}")
print(f"Next round: {round2.hasNext()}")
print(f"Remaining: {list(round2.getNextRoundInput().contestants.keys())}")
print("  → C (32%) eliminated: A+B = 68% ≥ 66.7% (⅔)")

In [ ]:
# --- Round 3 (A, B) ---
round3 = election.runNextIteration(
    iterationInput=round2.getNextRoundInput().with_votes({'A': 52, 'B': 48})
)

print("=== Round 3 ===")
print(round3.getContestantsVotesAfterPossibleCoalitions().to_string())
print(f"\nWinner: {round3.getWinner().name}")
print(f"  → A reaches 52% ≥ 52% (threshold) → election finished after {election.getNumberOfIterations()} rounds")

### Automatic run

`Election.run()` executes all rounds automatically. With random votes and a fixed `seed` the result is reproducible:

In [ ]:
config_auto = ElectionConfig(
    constituencies_config=cc,
    participating_parties=['A', 'B', 'C', 'D', 'E'],
    parliament_majority_margin=SuperMajorityMargin(2.0, MarginUnit.PERCENT),
    draw_lots_strategy=DrawLotsStrategy.MARGINAL_LEAD,
    seed=47,
)

election_auto = Election(electionConfig=config_auto)
final = election_auto.run()

print(f"Winner: {final.getWinner().name}  ({election_auto.getNumberOfIterations()} rounds)")
for i, r in enumerate(election_auto.iterations, 1):
    votes = r.getContestantsVotesAfterPossibleCoalitions()
    print(f"  Round {i}: {votes.to_dict()}")

---
## Example 2: Coalition formation

Three parties, winning threshold **55 %**. No party reaches the threshold on its own.
After the first round, B and C form a coalition.

| Party | Votes | Share |
|-------|:-----:|:-----:|
| A     | 40    | 40 %  |
| B     | 35    | 35 %  |
| C     | 25    | 25 %  |

B + C together: **60 % ≥ 55 %** → the coalition wins immediately, no second round needed.

In [ ]:
config_c = ElectionConfig(
    constituencies_config=cc,
    participating_parties=['A', 'B', 'C'],
    parliament_majority_margin=SuperMajorityMargin(5.0, MarginUnit.PERCENT),
    draw_lots_strategy=DrawLotsStrategy.MARGINAL_LEAD,
)

election_c = Election(electionConfig=config_c)
round_c = election_c.start(votes={'A': 40, 'B': 35, 'C': 25})

print(f"Threshold: {config_c.getParliamentMajorityPercent():.0f} %")
print(round_c.getContestantsVotesAfterPossibleCoalitions().to_string())
print(f"\nWinner before coalition: {round_c.getWinner()}")

# Form coalition B+C
round_c.formCoalition("B+C", ["B", "C"])

print(f"\nAfter forming coalition B+C (35+25=60%):")
print(round_c.getContestantsVotesAfterPossibleCoalitions().to_string())
print(f"\nWinner: {round_c.getWinner().name}")
print(f"Total rounds: {election_c.getNumberOfIterations()}")

---
## Example 3: Special case – drawing of lots

Two parties, winning threshold **52 %**. Party A receives 51 votes in both rounds — never enough to win, no reduction possible.
After two inconclusive rounds with the same two parties, the third round is decided by lot.

With `DrawLotsStrategy.MARGINAL_LEAD` the marginal vote difference is treated as a random outcome — the party that happened to receive slightly more votes (here A with 51 > 49) wins.

In [ ]:
config_l = ElectionConfig(
    constituencies_config=cc,
    participating_parties=['A', 'B'],
    parliament_majority_margin=SuperMajorityMargin(2.0, MarginUnit.PERCENT),
    draw_lots_strategy=DrawLotsStrategy.MARGINAL_LEAD,
)

election_l = Election(electionConfig=config_l)

votes_ab = {'A': 51, 'B': 49}   # A < 52% → no winner

# Round 1
r1_l = election_l.start(votes=votes_ab)

# Round 2
r2_l = election_l.runNextIteration(
    iterationInput=r1_l.getNextRoundInput().with_votes(votes_ab)
)

# Round 3 – DrawOfLots triggered automatically
r3_l = election_l.runNextIteration(
    iterationInput=r2_l.getNextRoundInput().with_votes(votes_ab)
)

for i, r in enumerate([r1_l, r2_l, r3_l], 1):
    kind = type(r).__name__
    print(f"Round {i} ({kind}): Winner={r.getWinner()}")

print(f"\nLot triggered: {isinstance(r3_l, DrawOfLots)}")
print(f"Winner: {r3_l.getWinner().name}  (has the most votes: {votes_ab['A']} > {votes_ab['B']})")

---

## `DrawLotsStrategy` — Both variants compared

When two parties fail to produce a winner in **two consecutive voting rounds**, the **third round** is decided by lot.

- **`RANDOM`** *(default)*: uniform random draw — the winner is selected at random.
- **`MARGINAL_LEAD`**: The marginal vote difference is treated as a random outcome — the party that happened to receive slightly more votes wins.

In [ ]:
votes_ab = {'A': 51, 'B': 49}   # A = 51% < 52% → no winner

for strategy in [DrawLotsStrategy.RANDOM, DrawLotsStrategy.MARGINAL_LEAD]:
    cfg = ElectionConfig(
        constituencies_config=cc,
        participating_parties=['A', 'B'],
        ballot_majority_margin=SuperMajorityMargin(2.0, MarginUnit.PERCENT),
        parliament_majority_margin=SuperMajorityMargin(2.0, MarginUnit.PERCENT),
        draw_lots_strategy=strategy,
        seed=7,
    )
    e = Election(electionConfig=cfg)
    r1 = e.start(votes=votes_ab)                                            # Round 1: no winner
    r2 = e.runNextIteration(r1.getNextRoundInput().with_votes(votes_ab))    # Round 2: no winner → lot
    r3 = e.runNextIteration(r2.getNextRoundInput().with_votes(votes_ab))    # Round 3: DrawOfLots
    print(f"{strategy.name:26s} → Winner: {e.getWinner().name}  {'(by lot)' if r3.wasDecidedByLot() else ''}")

---

## `ballot_majority_margin` — Ballot threshold

`ballot_majority_margin` sets the vote share a contestant must exceed in a **single voting round** to win that round. It is independent of `parliament_majority_margin`, which governs the governing-majority seat share in parliament.

Default: `SuperMajorityMargin(2.0, MarginUnit.PERCENT)` → threshold 52%. Can also be specified in seats (`MarginUnit.SEATS`), which is more practical for larger configurations (see notebook `global_configuration`).

In [ ]:
# Party A receives 53% of the votes.

# ballot_majority_margin = 2% (default): threshold 52% → A wins immediately
config_bm_low = ElectionConfig(
    constituencies_config=cc,
    participating_parties=['A', 'B'],
    ballot_majority_margin=SuperMajorityMargin(2.0, MarginUnit.PERCENT),
    parliament_majority_margin=SuperMajorityMargin(2.0, MarginUnit.PERCENT),
)
r = Election(electionConfig=config_bm_low).start(votes={'A': 53, 'B': 47})
print(f"ballot_majority_margin=2%  →  threshold {config_bm_low.getBallotMajorityPercent():.0f}%  →  A=53%  →  winner: {r.getWinner()}")

# ballot_majority_margin = 5% (higher): threshold 55% → A falls short
config_bm_high = ElectionConfig(
    constituencies_config=cc,
    participating_parties=['A', 'B'],
    ballot_majority_margin=SuperMajorityMargin(5.0, MarginUnit.PERCENT),
    parliament_majority_margin=SuperMajorityMargin(5.0, MarginUnit.PERCENT),
)
r2 = Election(electionConfig=config_bm_high).start(votes={'A': 53, 'B': 47})
w = r2.getWinner()
print(f"ballot_majority_margin=5%  →  threshold {config_bm_high.getBallotMajorityPercent():.0f}%  →  A=53%  →  winner: {w if w else 'no winner → next round needed'}")

---

## Vote injection with multiple constituencies

For multiple constituencies pass the `votes` argument as a nested dict:
`{constituency: {party: votes}}`. Votes are summed across all constituencies to obtain total votes per party.

In [ ]:
# 2-constituency configuration
cc2 = ConstituenciesConfig.from_dataframe(pd.DataFrame({
    'constituency_name': ['C1', 'C2'],
    'constituency_size': [100, 100],
    'turnout_percent':   [100.0, 100.0],
}))

config_mc = ElectionConfig(
    constituencies_config=cc2,
    participating_parties=['A', 'B', 'C'],
    parliament_majority_margin=SuperMajorityMargin(5.0, MarginUnit.PERCENT),
)

election_mc = Election(electionConfig=config_mc)
round_mc = election_mc.start(votes={
    'C1': {'A': 28, 'B': 22, 'C': 50},   # C1: C leads
    'C2': {'A': 40, 'B': 30, 'C': 30},   # C2: A leads
})

print("Votes per constituency:")
print(round_mc.vote_matrix.getVotes())
votes = round_mc.getContestantsVotesAfterPossibleCoalitions()
total = votes.sum()
print(f"\nTotal votes:  {votes.to_dict()}")
print(f"Vote shares: { {k: f'{v/total*100:.1f}%' for k, v in votes.items()} }")
w_mc = round_mc.getWinner()
print(f"Winner: {w_mc.name if w_mc else 'no winner'}")